# 🤖 Social Media Post AI Agent
**Workflow:** User request → Company/Product lookup → Reports → Style Guide → Post Generation → Scheduling → Logging

**Setup:** Fill in your API keys in `config.py` before running.

## 📦 Step 0: Imports & Config

In [ ]:
import os, json, re, datetime
from pathlib import Path

# ── Local modules (same folder as this notebook) ──────────────────────────────
from agent_storage   import Storage
from agent_research  import ResearchAgent
from agent_posts     import PostAgent
from agent_schedule  import ScheduleAgent
from agent_logger    import Logger
from agent_utils     import confirm, multiline_input, print_receipt

print("✅ Imports OK")

NameError: name '_instagram_instructions' is not defined

## 🔑 Step 1: API Keys

In [ ]:
# ── Paste your keys here (or load from .env) ───────────────────────────────────
OPENAI_API_KEY        = os.getenv("OPENAI_API_KEY",  "sk-...")   # GPT-4o for main agents
OPENAI_REVIEW_KEY     = os.getenv("OPENAI_API_KEY",  "sk-...")   # GPT-3.5 for A/B reviewer
SERPER_API_KEY        = os.getenv("SERPER_API_KEY",  "...")       # serper.dev – web search
GOOGLE_CALENDAR_ID    = os.getenv("GCAL_ID",         "primary")   # your calendar id
# Google Calendar credentials JSON path (from Google Cloud Console → Service Account)
GCAL_CREDENTIALS_JSON = os.getenv("GCAL_CREDENTIALS", "credentials.json")

print("✅ Keys loaded")

## 📁 Step 2: Storage Paths

In [ ]:
# Root of the project — change if your folder is elsewhere
PROJECT_ROOT = Path(".")   # same folder as this notebook
storage = Storage(PROJECT_ROOT / "AI Storage")
print(f"📂 AI Storage: {storage.root.resolve()}")

## 🗣️ Step 3: Parse User Request

In [ ]:
# ── EXAMPLE: change this string or make it an input() ─────────────────────────
USER_REQUEST = "Create 3 Instagram posts for Rita's Kiwi Melon in June + Schedule. Make it interactive please."

from agent_utils import parse_request
receipt = parse_request(USER_REQUEST)
print_receipt(receipt)

## 🔍 Step 4: Company & Product Lookup

In [ ]:
researcher = ResearchAgent(
    storage=storage,
    openai_key=OPENAI_API_KEY,
    serper_key=SERPER_API_KEY,
)

company_report, product_report = await researcher.resolve(
    company_name=receipt["company"],
    product_name=receipt["product"],
)

## 📖 Step 5: Review Reports (Optional)

In [ ]:
want_review = confirm("Would you like to review the Company or Product report before continuing? (y/N)")
if want_review:
    choice = input("Type 'company', 'product', or 'both': ").strip().lower()
    if choice in ("company", "both"):
        print(json.dumps(company_report, indent=2))
    if choice in ("product", "both"):
        print(json.dumps(product_report, indent=2))
    input("\nPress Enter when done reviewing (or edit the JSON files directly, then press Enter)…")

## 🎨 Step 6: Style Guide

In [ ]:
style_guide = await researcher.resolve_style_guide(
    company_report=company_report,
    product_report=product_report,
)

## ✅ Step 7: Confirm Receipt

In [ ]:
from agent_utils import interactive_receipt_editor
receipt = interactive_receipt_editor(receipt)
print_receipt(receipt)

## ✍️ Step 8: Generate Posts

In [ ]:
post_agent = PostAgent(
    openai_key=OPENAI_API_KEY,
    review_key=OPENAI_REVIEW_KEY,
)

posts = await post_agent.generate(
    receipt=receipt,
    company_report=company_report,
    product_report=product_report,
    style_guide=style_guide,
)

print(f"\n✅ Generated {len(posts)} post(s)")
for i, p in enumerate(posts, 1):
    print(f"\n--- Post {i} ---")
    print(p["caption"])

## 💾 Step 9: Save Posts

In [ ]:
output_dir = storage.save_posts(
    company=receipt["company"],
    product=receipt["product"],
    posts=posts,
    receipt=receipt,
)
print(f"📁 Posts saved to: {output_dir}")

## 📅 Step 10: Schedule Posts (Optional)

In [ ]:
if receipt.get("schedule"):
    scheduler = ScheduleAgent(
        openai_key=OPENAI_API_KEY,
        credentials_json=GCAL_CREDENTIALS_JSON,
        calendar_id=GOOGLE_CALENDAR_ID,
    )
    await scheduler.run(
        receipt=receipt,
        posts=posts,
        output_dir=output_dir,
    )
else:
    print("⏭️  Scheduling skipped.")

## 📝 Step 11: Write Log

In [ ]:
logger = Logger(storage)
log_path = logger.write(
    receipt=receipt,
    company_report=company_report,
    product_report=product_report,
    style_guide=style_guide,
    posts=posts,
    output_dir=output_dir,
)
print(f"\n🎉 All done! Log: {log_path}")